In [1]:
# Replace these placeholders with your actual values from the Azure AI Foundry Portal
AZURE_FOUNDRY_ENDPOINT = "https://foundrypvn.services.ai.azure.com/" # Found in project overview
MODEL_DEPLOYMENT_NAME = "gpt-4o-mini"                                            # Your model deployment name
AZURE_FOUNDRY_API_KEY = "E5dTYVgZuoZW7ttpD7TkSR0r3nvKNrdPwY9g31mvF6HoQupiDeOfJQQJ99CEACHYHv6XJ3w3AAAAACOGgoe8"                        # Your project connection key

In [42]:
import os
import asyncio
# Core authentication classes for secure cloud access
from azure.identity import DefaultAzureCredential

# Official MAF SDK components for building intelligent workflows
from agent_framework.foundry import FoundryChatClient
from agent_framework import Agent

print("Python: Framework modules successfully initialized inside the Conda workspace.")

Python: Framework modules successfully initialized inside the Conda workspace.


In [43]:
# Use the token-based Azure credential that FoundryChatClient expects.
# If you still see a tenant-mismatch error, sign in to the tenant that owns the project:
#   az login --tenant <PROJECT_TENANT_ID>
credential = DefaultAzureCredential()

foundry_client = FoundryChatClient(
    project_endpoint=AZURE_FOUNDRY_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)

print(f"Python: Successfully mapped client to model '{MODEL_DEPLOYMENT_NAME}' via Azure AI Foundry backend.")

Python: Successfully mapped client to model 'gpt-4o-mini' via Azure AI Foundry backend.


In [44]:
agent = Agent(
    client=foundry_client,
    name="HelloAgent",
    instructions="You are a friendly assistant. Keep your answers brief.",
)

In [45]:
result = await agent.run("What is the capital of France?")
print(f"Agent: {result}")

Agent: The capital of France is Paris.


In [22]:
res=agent.run("Tell me a one-sentence fun fact.")
print(f"Agent: {getattr(res, 'text', res)}")
print(f"Full response object: {res}")
print(f"Response metadata: {getattr(res, 'metadata', {})}")
print(f"Response generation time: {getattr(res, 'generation_time', 'N/A')} seconds")
print(f"Response tokens used: {getattr(res, 'tokens_used', 'N/A')}")
print(f"Response model: {getattr(res, 'model', 'N/A')}")


Agent: <coroutine object AgentTelemetryLayer._trace_agent_invocation.<locals>._run at 0x0000027FF10F90E0>
Full response object: <coroutine object AgentTelemetryLayer._trace_agent_invocation.<locals>._run at 0x0000027FF10F90E0>
Response metadata: {}
Response generation time: N/A seconds
Response tokens used: N/A
Response model: N/A


C:\Users\admin\AppData\Local\Temp\ipykernel_18664\1257352327.py:1: RuntimeWarning: coroutine 'AgentTelemetryLayer._trace_agent_invocation.<locals>._run' was never awaited
  res=agent.run("Tell me a one-sentence fun fact.")


In [ ]:
async def test_baseline_connection():
    # Define a generic agent with no special constraints
    test_agent = Agent(
        name="TestAgent",
        client=foundry_client,
        instructions="You are a helpful assistant."
    )
    
    # Execute a simple, deterministic completion request
    result = await test_agent.run("What is the capital of France?")
    print(f"Agent Response: {result}")

# Run the cell verification
await test_agent_connection()

# --- .NET (C#) LINE-BY-LINE EQUIVALENT ---
# /*
# var dotnetTestAgent = new Agent("TestAgent", dotnetFoundryClient, "You are a helpful assistant.");
# var dotnetResult = await dotnetTestAgent.RunAsync("What is the capital of France?");
# Console.WriteLine($"Agent Response: {dotnetResult}");
# */

In [ ]:
# 1. DEFINE THE NATIVE PYTHON FUNCTION TO INTERACT WITH AN ERP DATABASE
@system_tool
def get_order_status(order_id: str) -> str:
    """Queries the internal enterprise e-commerce ERP database to fetch live shipment tracking telemetry."""
    if order_id == "ORD-9982":
        return "Status: Shipped. Carrier: FedEx. Current Hub: Chicago Distribution Center. ETA: Tomorrow by 5:00 PM."
    return f"Status: Processing. Order ID {order_id} found in ledger. Awaiting carrier pickup."

async def test_tool_execution():
    # 2. INSTANTIATE AGENT BINDING THE TOOL TO ITS RUNTIME EXTENSION
    support_agent = Agent(
        name="SupportTriager",
        client=foundry_client,
        instructions="You are a front-line e-commerce database investigator. Use tools to look up order information.",
        tools=[get_order_status]
    )
    
    # 3. ASK A QUESTION TRIGERRES THE TOOL DEFINITION
    tool_result = await support_agent.run("Where is my package ORD-9982?")
    print(f"Tool Agent Response:\n{tool_result}")

# Run the cell verification
await test_tool_execution()

# --- .NET (C#) LINE-BY-LINE EQUIVALENT ---
# /*
# // .NET requires method metadata decoration matching the Python function above
# [SystemTool("Queries the internal enterprise e-commerce ERP database to fetch live shipment tracking telemetry.")]
# public static string GetOrderStatus(string orderId) { ... }
# 
# var dotnetSupport = new Agent("SupportTriager", dotnetFoundryClient, "You are a database investigator.");
# dotnetSupport.RegisterTool(GetOrderStatus); // Explicit tool mapping
# var dotnetToolResult = await dotnetSupport.RunAsync("Where is my package ORD-9982?");
# */

In [ ]:
async def run_multi_agent_pipeline():
    # Re-declare Agent A (The Investigator)
    support_agent = Agent(
        name="SupportTriager",
        client=foundry_client,
        instructions="Extract exact logs from the order tool and pass the raw data downstream.",
        tools=[get_order_status]
    )

    # Declare Agent B (The Diplomatic Communicator)
    delivery_concierge = Agent(
        name="DeliveryConcierge",
        client=foundry_client,
        instructions="Take raw telemetry data and write an elegant, polite response to the customer. Do not call tools."
    )

    # 1. DEFINE THE MAF WORKFLOW WRAPPER
    ecom_workflow = SequentialWorkflow(name="Fulfillment_Flow")
    ecom_workflow.add_agent(support_agent)      # Step 1 executes first
    ecom_workflow.add_agent(delivery_concierge)  # Step 2 processes Step 1's output text

    # 2. RUN THE ENTIRE ECOSYSTEM
    customer_input = "Hi, check order ORD-9982. It hasn't arrived!"
    pipeline_result = await ecom_workflow.run(input_data=customer_input)
    print(f"Final Client Update:\n{pipeline_result}")

# Run the cell verification
await run_multi_agent_pipeline()

# --- .NET (C#) LINE-BY-LINE EQUIVALENT ---
# /*
# var dotnetWorkflow = new SequentialWorkflow("Fulfillment_Flow");
# dotnetWorkflow.AddAgent(dotnetSupport);
# dotnetWorkflow.AddAgent(dotnetConcierge);
# var dotnetFinalResult = await dotnetWorkflow.RunAsync("Hi, check order ORD-9982.");
# */